# OpenAlex Publication-Context Preparation

## Purpose

This notebook prepares the OpenAlex publication dataset for use in GrantScopeAI.

Unlike NSF and CORDIS, OpenAlex does not represent individual funding awards. It provides publication and citation activity that can be used to measure research momentum across the selected AI-enabled chemistry and materials topics.

The cleaned OpenAlex dataset will remain at the **topic-year level**, with one row for each topic and year combination.

## Objectives

This notebook will:

- load and preserve the existing OpenAlex extraction;
- validate coverage across the selected topics and the 2021–2025 period;
- confirm one row per unique topic-year combination;
- standardize topic labels, years, publication counts, and citation metrics;
- create stable source and topic-year identifiers;
- document important methodological decisions and limitations;
- export a compact processed table for analysis, Tableau, and Streamlit;
- create a machine-readable validation summary.

## Expected completion criteria

The notebook is complete when:

- all eight selected topics are represented;
- each topic contains one record for every year from 2021 through 2025;
- the dataset contains 40 unique topic-year records;
- duplicate topic-year combinations have been resolved;
- publication metrics are numeric, non-negative, and documented;
- the cleaned and validation outputs are successfully exported;
- the notebook runs from beginning to end without errors.

In [1]:
from pathlib import Path
import pandas as pd


# Define the GrantScopeAI project folder
PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

OPENALEX_RAW_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Raw_Data"
    / "OpenAlex"
)


# Find available OpenAlex CSV files
openalex_raw_files = list(
    OPENALEX_RAW_DIR.glob("*.csv")
)

# Fallback in case the file was saved directly in Raw_Data
if not openalex_raw_files:
    openalex_raw_files = list(
        (PROJECT_ROOT / "Data" / "Raw_Data").glob(
            "*openalex*.csv"
        )
    )

if not openalex_raw_files:
    raise FileNotFoundError(
        "No OpenAlex CSV file was found in "
        f"{OPENALEX_RAW_DIR}"
    )


# Select the most recently modified OpenAlex CSV
openalex_raw_path = max(
    openalex_raw_files,
    key=lambda path: path.stat().st_mtime
)


# Preserve the imported data and create a working copy
openalex_raw_df = pd.read_csv(
    openalex_raw_path,
    dtype="string",
    low_memory=False
)

openalex_df = openalex_raw_df.copy()


print("OpenAlex file loaded:")
print(openalex_raw_path)

print(
    "\nDataset shape:",
    openalex_df.shape
)

print("\nColumns:")
print(openalex_df.columns.tolist())

display(openalex_df.head())

OpenAlex file loaded:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\openalex_topic_year_counts_2021_2025_2026-08-01.csv

Dataset shape: (40, 5)

Columns:
['topic', 'publication_year', 'publication_count', 'search_query', 'source']


,topic,publication_year,publication_count,search_query,source
0,AI-enabled chemistry,2021,40651,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
1,AI-enabled chemistry,2022,54337,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
2,AI-enabled chemistry,2023,96605,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
3,AI-enabled chemistry,2024,112587,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
4,AI-enabled chemistry,2025,142246,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex


In [2]:
# Validate the raw OpenAlex topic-year structure

openalex_topic_count = openalex_df["topic"].nunique(dropna=True)
openalex_year_count = openalex_df["publication_year"].nunique(dropna=True)

openalex_duplicate_topic_years = (
    openalex_df[
        ["topic", "publication_year"]
    ]
    .duplicated()
    .sum()
)

openalex_missing_values_df = (
    openalex_df.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

openalex_topic_coverage_df = (
    openalex_df.groupby("topic")
    .agg(
        row_count=("publication_year", "size"),
        unique_years=("publication_year", "nunique")
    )
    .reset_index()
    .sort_values("topic")
)

print(
    "Total rows:",
    f"{len(openalex_df):,}"
)

print(
    "Unique topics:",
    openalex_topic_count
)

print(
    "Unique publication years:",
    openalex_year_count
)

print(
    "Duplicate topic-year combinations:",
    openalex_duplicate_topic_years
)

print(
    "\nPublication years:",
    sorted(
        openalex_df["publication_year"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nTopics:")
for topic in sorted(
    openalex_df["topic"]
    .dropna()
    .unique()
    .tolist()
):
    print(f"- {topic}")

print("\nMissing values:")
display(openalex_missing_values_df)

print("\nCoverage by topic:")
display(openalex_topic_coverage_df)

Total rows: 40
Unique topics: 8
Unique publication years: 5
Duplicate topic-year combinations: 0

Publication years: ['2021', '2022', '2023', '2024', '2025']

Topics:
- AI-enabled catalysis
- AI-enabled chemistry
- AI-enabled materials
- Autonomous laboratories
- Cheminformatics
- Materials informatics
- Molecular machine learning
- Reaction prediction

Missing values:


,column,missing_count
0,topic,0
1,publication_year,0
2,publication_count,0
3,search_query,0
4,source,0



Coverage by topic:


,topic,row_count,unique_years
0,AI-enabled catalysis,5,5
1,AI-enabled chemistry,5,5
2,AI-enabled materials,5,5
3,Autonomous laboratories,5,5
4,Cheminformatics,5,5
5,Materials informatics,5,5
6,Molecular machine learning,5,5
7,Reaction prediction,5,5


In [3]:
# Standardize OpenAlex fields and create a stable topic-year key

openalex_clean_df = openalex_df.copy()

openalex_clean_df["topic_clean"] = (
    openalex_clean_df["topic"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

openalex_clean_df["publication_year"] = pd.to_numeric(
    openalex_clean_df["publication_year"],
    errors="coerce"
).astype("Int64")

openalex_clean_df["publication_count"] = pd.to_numeric(
    openalex_clean_df["publication_count"],
    errors="coerce"
).astype("Int64")

openalex_clean_df["search_query"] = (
    openalex_clean_df["search_query"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

openalex_clean_df["source"] = "OpenAlex"

openalex_clean_df["topic_slug"] = (
    openalex_clean_df["topic_clean"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

openalex_clean_df["topic_year_key"] = (
    "OPENALEX_"
    + openalex_clean_df["topic_slug"].str.upper()
    + "_"
    + openalex_clean_df["publication_year"].astype("string")
)

invalid_year_count = (
    openalex_clean_df["publication_year"].isna().sum()
)

invalid_publication_count = (
    openalex_clean_df["publication_count"].isna().sum()
)

negative_publication_count = (
    openalex_clean_df["publication_count"].lt(0).sum()
)

duplicate_key_count = (
    openalex_clean_df["topic_year_key"].duplicated().sum()
)

print("Rows:", len(openalex_clean_df))
print("Invalid years:", invalid_year_count)
print("Invalid publication counts:", invalid_publication_count)
print("Negative publication counts:", negative_publication_count)
print("Duplicate topic-year keys:", duplicate_key_count)

display(
    openalex_clean_df[
        [
            "topic_year_key",
            "topic_clean",
            "publication_year",
            "publication_count",
            "source"
        ]
    ].head(10)
)

Rows: 40
Invalid years: 0
Invalid publication counts: 0
Negative publication counts: 0
Duplicate topic-year keys: 0


,topic_year_key,topic_clean,publication_year,publication_count,source
0,OPENALEX_AI_ENABLED_CHEMISTRY_2021,AI-enabled chemistry,2021,40651,OpenAlex
1,OPENALEX_AI_ENABLED_CHEMISTRY_2022,AI-enabled chemistry,2022,54337,OpenAlex
2,OPENALEX_AI_ENABLED_CHEMISTRY_2023,AI-enabled chemistry,2023,96605,OpenAlex
3,OPENALEX_AI_ENABLED_CHEMISTRY_2024,AI-enabled chemistry,2024,112587,OpenAlex
4,OPENALEX_AI_ENABLED_CHEMISTRY_2025,AI-enabled chemistry,2025,142246,OpenAlex
5,OPENALEX_AI_ENABLED_MATERIALS_2021,AI-enabled materials,2021,107790,OpenAlex
6,OPENALEX_AI_ENABLED_MATERIALS_2022,AI-enabled materials,2022,146376,OpenAlex
7,OPENALEX_AI_ENABLED_MATERIALS_2023,AI-enabled materials,2023,250712,OpenAlex
8,OPENALEX_AI_ENABLED_MATERIALS_2024,AI-enabled materials,2024,295260,OpenAlex
9,OPENALEX_AI_ENABLED_MATERIALS_2025,AI-enabled materials,2025,386893,OpenAlex


## Measuring publication momentum

OpenAlex is used as a publication-context source rather than as a grant dataset.

The analytical grain remains one row per topic and year. Publication momentum is measured using:

- annual publication count;
- year-over-year change;
- year-over-year percentage growth;
- an index using 2021 as the baseline value of 100.

These indicators allow trends to be compared within each topic over time.

Publication counts are not added across topics because the underlying search queries may overlap. Growth in publication volume is also treated as evidence of research activity rather than proof of research quality, novelty, or future funding availability.

In [4]:
# Calculate publication-growth indicators by topic

openalex_clean_df = (
    openalex_clean_df
    .sort_values(
        ["topic_clean", "publication_year"]
    )
    .reset_index(drop=True)
)

openalex_clean_df["publication_change"] = (
    openalex_clean_df
    .groupby("topic_clean")["publication_count"]
    .diff()
    .astype("Int64")
)

openalex_clean_df["publication_growth_yoy_pct"] = (
    openalex_clean_df
    .groupby("topic_clean")["publication_count"]
    .pct_change(fill_method=None)
    .mul(100)
    .round(2)
)

topic_baseline_counts = (
    openalex_clean_df
    .groupby("topic_clean")["publication_count"]
    .transform("first")
)

openalex_clean_df["publication_index_2021"] = (
    openalex_clean_df["publication_count"]
    .div(topic_baseline_counts)
    .mul(100)
    .round(2)
)

print(
    "Rows with year-over-year growth:",
    openalex_clean_df[
        "publication_growth_yoy_pct"
    ].notna().sum()
)

print(
    "Rows without year-over-year growth:",
    openalex_clean_df[
        "publication_growth_yoy_pct"
    ].isna().sum()
)

display(
    openalex_clean_df[
        [
            "topic_clean",
            "publication_year",
            "publication_count",
            "publication_change",
            "publication_growth_yoy_pct",
            "publication_index_2021"
        ]
    ].head(10)
)

Rows with year-over-year growth: 32
Rows without year-over-year growth: 8


,topic_clean,publication_year,publication_count,publication_change,publication_growth_yoy_pct,publication_index_2021
0,AI-enabled catalysis,2021,6094,<NA>,<NA>,100.0
1,AI-enabled catalysis,2022,8449,2355,38.64,138.64
2,AI-enabled catalysis,2023,18137,9688,114.66,297.62
3,AI-enabled catalysis,2024,25744,7607,41.94,422.45
4,AI-enabled catalysis,2025,39893,14149,54.96,654.63
5,AI-enabled chemistry,2021,40651,<NA>,<NA>,100.0
6,AI-enabled chemistry,2022,54337,13686,33.67,133.67
7,AI-enabled chemistry,2023,96605,42268,77.79,237.64
8,AI-enabled chemistry,2024,112587,15982,16.54,276.96
9,AI-enabled chemistry,2025,142246,29659,26.34,349.92


## Summarizing topic-level growth

A topic-level summary is created to compare publication activity between 2021 and 2025.

For each topic, the summary includes:

- publication counts for each year;
- absolute growth from 2021 to 2025;
- total percentage growth across the period;
- compound annual growth rate over the four annual intervals.

These metrics should be interpreted together. Topics with small starting values can show very high percentage growth, while larger topics may show more substantial absolute growth but lower percentage growth.

In [5]:
# Summarize publication momentum from 2021 to 2025

openalex_topic_summary_df = (
    openalex_clean_df.pivot(
        index="topic_clean",
        columns="publication_year",
        values="publication_count"
    )
    .reset_index()
)

openalex_topic_summary_df = (
    openalex_topic_summary_df.rename(
        columns={
            2021: "publications_2021",
            2022: "publications_2022",
            2023: "publications_2023",
            2024: "publications_2024",
            2025: "publications_2025"
        }
    )
)

openalex_topic_summary_df["absolute_growth_2021_2025"] = (
    openalex_topic_summary_df["publications_2025"]
    - openalex_topic_summary_df["publications_2021"]
)

openalex_topic_summary_df["total_growth_2021_2025_pct"] = (
    (
        openalex_topic_summary_df["publications_2025"]
        / openalex_topic_summary_df["publications_2021"]
        - 1
    )
    * 100
).round(2)

# Four annual growth periods occur between 2021 and 2025
openalex_topic_summary_df["cagr_2021_2025_pct"] = (
    (
        openalex_topic_summary_df["publications_2025"]
        / openalex_topic_summary_df["publications_2021"]
    )
    ** (1 / 4)
    - 1
).mul(100).round(2)

openalex_topic_summary_df = (
    openalex_topic_summary_df
    .sort_values(
        "cagr_2021_2025_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(openalex_topic_summary_df)

publication_year,topic_clean,publications_2021,publications_2022,publications_2023,publications_2024,publications_2025,absolute_growth_2021_2025,total_growth_2021_2025_pct,cagr_2021_2025_pct
0,Autonomous laboratories,75,119,273,440,859,784,1045.33,83.96
1,AI-enabled catalysis,6094,8449,18137,25744,39893,33799,554.63,59.96
2,AI-enabled materials,107790,146376,250712,295260,386893,279103,258.93,37.64
3,AI-enabled chemistry,40651,54337,96605,112587,142246,101595,249.92,36.77
4,Reaction prediction,197,256,484,505,576,379,192.39,30.76
5,Molecular machine learning,189,263,633,633,549,360,190.48,30.55
6,Materials informatics,398,511,807,836,1021,623,156.53,26.56
7,Cheminformatics,2511,2997,4606,4590,4975,2464,98.13,18.64


## Validating topic-year completeness

Before export, the cleaned OpenAlex table is checked to confirm that it provides complete and reliable publication context.

The validation confirms:

- 40 unique topic-year records;
- eight topics;
- complete annual coverage from 2021 through 2025;
- no duplicate topic-year combinations or keys;
- no missing or negative publication counts;
- a valid 2021 baseline index for every topic.

The eight 2021 records do not contain year-over-year growth values because no earlier year is available in the dataset. This is expected rather than treated as missing-data error.

In [6]:
# Create the final OpenAlex data-quality validation summary

expected_topics = 8
expected_years = {2021, 2022, 2023, 2024, 2025}

observed_years = set(
    openalex_clean_df["publication_year"]
    .dropna()
    .astype(int)
    .tolist()
)

topic_year_coverage_df = (
    openalex_clean_df.groupby("topic_clean")
    .agg(
        row_count=("publication_year", "size"),
        unique_year_count=("publication_year", "nunique"),
        minimum_year=("publication_year", "min"),
        maximum_year=("publication_year", "max")
    )
    .reset_index()
)

topics_with_incomplete_coverage = (
    topic_year_coverage_df[
        "unique_year_count"
    ].ne(5).sum()
)

invalid_baseline_index_count = (
    openalex_clean_df.loc[
        openalex_clean_df["publication_year"].eq(2021),
        "publication_index_2021"
    ]
    .ne(100)
    .sum()
)

openalex_validation_summary_df = pd.DataFrame({
    "check": [
        "Total topic-year rows",
        "Unique topics",
        "Unique publication years",
        "Expected years present",
        "Duplicate topic-year combinations",
        "Duplicate topic-year keys",
        "Missing topic labels",
        "Missing publication years",
        "Missing publication counts",
        "Negative publication counts",
        "Topics with incomplete year coverage",
        "Invalid 2021 baseline indexes",
        "Rows with year-over-year growth",
        "Rows without year-over-year growth"
    ],
    "result": [
        len(openalex_clean_df),
        openalex_clean_df["topic_clean"].nunique(),
        openalex_clean_df["publication_year"].nunique(),
        observed_years == expected_years,
        openalex_clean_df[
            ["topic_clean", "publication_year"]
        ].duplicated().sum(),
        openalex_clean_df[
            "topic_year_key"
        ].duplicated().sum(),
        openalex_clean_df[
            "topic_clean"
        ].isna().sum(),
        openalex_clean_df[
            "publication_year"
        ].isna().sum(),
        openalex_clean_df[
            "publication_count"
        ].isna().sum(),
        openalex_clean_df[
            "publication_count"
        ].lt(0).sum(),
        topics_with_incomplete_coverage,
        invalid_baseline_index_count,
        openalex_clean_df[
            "publication_growth_yoy_pct"
        ].notna().sum(),
        openalex_clean_df[
            "publication_growth_yoy_pct"
        ].isna().sum()
    ]
})

display(openalex_validation_summary_df)

print("\nCoverage by topic:")
display(topic_year_coverage_df)

,check,result
0,Total topic-year rows,40
1,Unique topics,8
2,Unique publication years,5
3,Expected years present,True
4,Duplicate topic-year combinations,0
5,Duplicate topic-year keys,0
6,Missing topic labels,0
7,Missing publication years,0
8,Missing publication counts,0
9,Negative publication counts,0



Coverage by topic:


,topic_clean,row_count,unique_year_count,minimum_year,maximum_year
0,AI-enabled catalysis,5,5,2021,2025
1,AI-enabled chemistry,5,5,2021,2025
2,AI-enabled materials,5,5,2021,2025
3,Autonomous laboratories,5,5,2021,2025
4,Cheminformatics,5,5,2021,2025
5,Materials informatics,5,5,2021,2025
6,Molecular machine learning,5,5,2021,2025
7,Reaction prediction,5,5,2021,2025


In [7]:
# Export the cleaned OpenAlex datasets

OPENALEX_PROCESSED_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Processed_Data"
    / "OpenAlex"
)

OPENALEX_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Preserve the extraction date recorded in the raw filename
extraction_date_match = openalex_raw_path.stem.split("_")[-1]

openalex_clean_df["extraction_date"] = pd.to_datetime(
    extraction_date_match,
    errors="coerce"
)


openalex_clean_output_path = (
    OPENALEX_PROCESSED_DIR
    / "openalex_topic_year_clean_2021_2025.csv"
)

openalex_topic_summary_output_path = (
    OPENALEX_PROCESSED_DIR
    / "openalex_topic_momentum_summary_2021_2025.csv"
)

openalex_validation_output_path = (
    OPENALEX_PROCESSED_DIR
    / "openalex_cleaning_validation_summary.csv"
)


openalex_clean_df.to_csv(
    openalex_clean_output_path,
    index=False,
    encoding="utf-8-sig"
)

openalex_topic_summary_df.to_csv(
    openalex_topic_summary_output_path,
    index=False,
    encoding="utf-8-sig"
)

openalex_validation_summary_df.to_csv(
    openalex_validation_output_path,
    index=False,
    encoding="utf-8-sig"
)


for label, path in {
    "Clean topic-year dataset": openalex_clean_output_path,
    "Topic momentum summary": openalex_topic_summary_output_path,
    "Validation summary": openalex_validation_output_path
}.items():

    print(f"\n{label}:")
    print(path)
    print(
        "File size:",
        f"{path.stat().st_size / 1_000_000:.3f} MB"
    )


Clean topic-year dataset:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\OpenAlex\openalex_topic_year_clean_2021_2025.csv
File size: 0.009 MB

Topic momentum summary:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\OpenAlex\openalex_topic_momentum_summary_2021_2025.csv
File size: 0.001 MB

Validation summary:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\OpenAlex\openalex_cleaning_validation_summary.csv
File size: 0.000 MB


## OpenAlex preparation summary

The OpenAlex publication-context dataset has been validated, standardized, and exported for use in GrantScopeAI.

### Final dataset

- 40 unique topic-year records;
- eight selected research topics;
- complete annual coverage from 2021 through 2025;
- no duplicate topic-year combinations;
- no missing or negative publication counts.

### Analytical role

OpenAlex remains a separate topic-year table rather than being merged directly into individual NSF or CORDIS grant records.

Publication momentum is represented through:

- annual publication counts;
- year-over-year growth;
- a 2021 baseline index;
- total growth and compound annual growth from 2021 to 2025.

The results indicate research activity and topic momentum. They are not treated as direct measures of research quality, novelty, or future funding potential.

### Outputs

The notebook exports:

- a cleaned topic-year dataset;
- a topic-level momentum summary;
- a data-quality validation summary.

All outputs are small enough to be committed to GitHub and used in Python analysis, Tableau, Streamlit, and later cross-source integration.